# Phase 2 — SFT LoRA puis DPO de Qwen3-1.7B-Base

Fine-tuning supervisé (SFT) de `Qwen/Qwen3-1.7B-Base` sur le dataset anonymisé de la phase 1, avec LoRA, puis alignement par préférences (DPO) du modèle SFT.

- **Prérequis** : GPU, fichiers `data/anonymized/sft_*_{train,validation,test}.anonymized.jsonl` produits par `data_prep_MediQAl` (français, réponse unique), `data_prep_SFT_MediQAl_mcqm` (français, réponses multiples) et `data_prep_SFT_UltraMedical` (anglais). Les QCM français contiennent les propositions A à E, puis une consigne sur le nombre de bonnes réponses (une seule ou plusieurs) ; la réponse attendue donne la ou les lettres et leur texte.
- **Prérequis DPO** : fichiers `data/anonymized/dpo_mediqal_mcqm_*` et `dpo_ultramedical_*` produits par `data_prep_DPO_MediQAl_mcqm` et `data_prep_DPO_UltraMedical`.
- **Étapes** : données → modèle → LoRA → entraînement SFT → validation intermédiaire → DPO → validation DPO → modèle final fusionné.
- **Suivi** : MLflow sur DagsHub (`MLFLOW_BACKEND = "dagshub"`), consultable de partout et quelle que soit la machine d'entraînement. En local (`"local"`) : `mlflow ui --backend-store-uri sqlite:///mlflow.db` depuis la racine du projet.

In [1]:
# Installation (Colab ou machine GPU) : décommenter si besoin
# %pip install -q torch transformers peft trl datasets accelerate mlflow dagshub pandas

## 1. Configuration

In [1]:
import functools
import gc
import json
import math
import re
from collections import Counter
from pathlib import Path

import mlflow
import pandas as pd
import torch
from datasets import DatasetDict, concatenate_datasets, load_dataset
from IPython.display import display
from peft import LoraConfig, PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer

from projet_14.mediqal import LETTERS, answer_letters

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
# Données SFT de la phase 1, anonymisées : fichier -> langue
# Le SFT couvre les formats et les langues du DPO (section 7) : réponses uniques et listes en français, anglais
SFT_DATASETS = {
    "sft_mediqal_mcqu": "fr",   # QCM à réponse unique, avec cas clinique
    "sft_mediqal_mcqm": "fr",   # QCM à réponses multiples (liste), avec et sans cas clinique
    "sft_ultramedical": "en",   # questions et réponses rédigées en anglais
}

# Suivi MLflow : "dagshub" (serveur distant) ou "local" (fichier ../mlflow.db)
MLFLOW_BACKEND = "dagshub"
DAGSHUB_REPO_OWNER = "obiyohan"
DAGSHUB_REPO_NAME = "my-ml-mlflow"

# Petit run pour valider la pipeline avant l'entraînement complet
SMALL_RUN = False  # True pour un run rapide sur 100 exemples, False pour le run complet

SEED = 42
# Version des données de la phase 1 : dans le nom des runs, pour ne pas écraser les checkpoints d'une version à l'autre
# v1 : propositions absentes du prompt ; v2 : propositions dans le prompt, répartition par cas clinique,
#      DPO anglais équilibré sur la longueur ; v3 : consigne « une seule / plusieurs propositions exactes »
DATA_VERSION = "v3"
RUN_NAME = f"sft-lora-r16-{DATA_VERSION}-seed{SEED}" + ("-small" if SMALL_RUN else "")
OUTPUT_DIR = Path("../outputs") / RUN_NAME  # checkpoints et adaptateur final

set_seed(SEED)
if torch.cuda.is_available():
    device, bf16 = "cuda", torch.cuda.is_bf16_supported()  # bf16 absent sur T4 : fp16 à la place
elif torch.xpu.is_available():
    device, bf16 = "xpu", torch.xpu.is_bf16_supported()  # GPU Intel Arc
else:
    device, bf16 = "cpu", False
dtype = torch.bfloat16 if bf16 else torch.float16 if device != "cpu" else torch.float32
print(f"device={device}, dtype={dtype}")

if device == "xpu":
    # GPU Intel sous Windows : deux contournements
    # 1. Pas de compilateur C pour Triton : TRL calcule la loss en PyTorch pur
    import trl.kernels.chunked_logprob as chunked_logprob
    chunked_logprob._TRITON_AVAILABLE = False
    # 2. La norme des gradients en mode « foreach » fait planter le GPU : calcul tenseur par tenseur
    torch.nn.utils.clip_grad_norm_ = functools.partial(torch.nn.utils.clip_grad.clip_grad_norm_, foreach=False)

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device=cuda, dtype=torch.bfloat16


## 2. Données : train / validation / test

In [3]:
# Format prompt / completion : la loss ne porte que sur la réponse de l'assistant
def to_prompt_completion(record):
    return {"prompt": record["messages"][:-1], "completion": record["messages"][-1:]}


# Chaque source garde son découpage train / validation / test ; les sources sont ensuite mélangées
parts = {split: [] for split in ("train", "validation", "test")}
for stem, lang in SFT_DATASETS.items():
    files = {split: f"../data/anonymized/{stem}_{split}.anonymized.jsonl" for split in parts}
    ds = load_dataset("json", data_files=files).select_columns(["messages"])
    for split, part in ds.items():
        parts[split].append(part.add_column("source", [stem] * len(part)).add_column("lang", [lang] * len(part)))

splits = DatasetDict({split: concatenate_datasets(p).shuffle(seed=SEED) for split, p in parts.items()})
splits = splits.map(to_prompt_completion, remove_columns=["messages"])

# Petit run : sous-échantillon (toutes sources mélangées)
if SMALL_RUN:
    for name, size in {"train": 50, "validation": 15, "test": 24}.items():
        splits[name] = splits[name].select(range(size))

pd.DataFrame({name: pd.Series(ds["source"]).value_counts() for name, ds in splits.items()}).fillna(0).astype(int)

Map: 100%|██████████| 2180/2180 [00:00<00:00, 13343.89 examples/s]


,train,validation,test
sft_mediqal_mcqm,2400,380,931
sft_mediqal_mcqu,2454,342,749
sft_ultramedical,1500,300,500


## 3. Tokenizer et modèle

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Gabarit ChatML, avec une réponse de l'assistant terminée par <|endoftext|> (eos_token) au lieu de <|im_end|>.
# Dans Qwen3-Base, <|im_start|> et <|im_end|> n'ont jamais été entraînés (embeddings à leur valeur d'initialisation)
# et LoRA ne modifie pas les embeddings : le modèle ne peut pas apprendre à produire <|im_end|>, donc à s'arrêter.
# <|endoftext|> a été appris au pré-entraînement : le modèle sait le prédire, et vLLM s'arrête dessus par défaut.
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] }}"
    "{% if message['role'] == 'assistant' %}{{ eos_token }}{% else %}{{ '<|im_end|>\n' }}{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)

example = splits["train"][0]
print(tokenizer.apply_chat_template(example["prompt"] + example["completion"], tokenize=False))

<|im_start|>system
Tu es un assistant médical expert. Réponds de façon précise et concise.<|im_end|>
<|im_start|>user
Un homme de 45 ans, dont le poids normal est de 60 kg, présente des adénopathies cervicales bilatérales, un amaigrissement de 7 kg en 6 mois, l'absence de fièvre, pas de sueurs nocturnes. L'examen permet de percevoir une rate débordant de 4 cm le rebord costal. Pas d'hépatomégalie, pas d'adénopathie dans les autres territoires accessibles à l'examen clinique. Le bilan objective : VS supérieure à 100 à la première heure, fibrinémie à 9 g/l, leucopénie à 3.000 globules blancs, pas de cellule anormale dans le sang, radiographie pulmonaire normale

La lymphographie était négative, et la laparotomie a objectivé une rate de 400 grammes, des ganglions du hile splénique et du pédicule hépatique envahis, un foie et une moelle osseuse histologiquement normaux. Le stade pathologique (PS) de ce patient est :

Propositions :
A. PS IIE
B. PS IIB
C. PS IIIA
D. PS IIIB
E. PS IVB

Une s

In [5]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
print(f"{model.num_parameters() / 1e9:.2f} B paramètres")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1758.43it/s]


1.72 B paramètres


## 4. LoRA

In [6]:
# Seules de petites matrices de rang r sont entraînées : le modèle de base reste figé
lora_config = LoraConfig(
    r=16,                         # rang des matrices d'adaptation
    lora_alpha=32,                # facteur d'échelle (souvent 2 × r)
    lora_dropout=0.05,
    target_modules="all-linear",  # couches d'attention et MLP
    task_type="CAUSAL_LM",
)

## 5. Entraînement SFT

In [7]:
# Suivi MLflow : un run par entraînement, hyperparamètres et métriques enregistrés
if MLFLOW_BACKEND == "dagshub":
    import dagshub

    # Jeton lu dans la variable d'environnement DAGSHUB_USER_TOKEN,
    # sinon connexion via le navigateur au premier lancement (jeton mémorisé ensuite)
    dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
else:
    mlflow.set_tracking_uri("sqlite:///../mlflow.db")
print(f"Suivi MLflow : {mlflow.get_tracking_uri()}")

mlflow.set_experiment("qwen3-1.7b-sft-lora")
mlflow.start_run(run_name=RUN_NAME)
mlflow.log_params({"model": MODEL_NAME, "data_version": DATA_VERSION, "small_run": SMALL_RUN, "device": device, "lora_r": lora_config.r, "lora_alpha": lora_config.lora_alpha, "lora_dropout": lora_config.lora_dropout})

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    run_name=RUN_NAME,
    seed=SEED,
    # Optimisation
    # Une époque : au run v2 (2 époques), eval_loss était au plus bas en fin d'époque 1 puis remontait légèrement
    num_train_epochs=1,
    max_steps=30 if SMALL_RUN else -1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # batch effectif = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,  # 3 % des étapes en montée progressive du learning rate
    max_length=1024,
    gradient_checkpointing=True,  # moins de mémoire GPU, un peu plus lent
    bf16=bf16,
    fp16=dtype == torch.float16,
    # Évaluation et checkpoints
    eval_strategy="steps",
    eval_steps=10 if SMALL_RUN else 100,
    save_steps=10 if SMALL_RUN else 100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # Logs
    logging_steps=5 if SMALL_RUN else 20,
    report_to="mlflow",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=splits["train"],
    eval_dataset=splits["validation"],
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

Accessing as ObiYohan

Initialized MLflow to track repo "obiyohan/my-ml-mlflow"

Repository obiyohan/my-ml-mlflow initialized!

Suivi MLflow : https://dagshub.com/obiyohan/my-ml-mlflow.mlflow


Truncating train dataset: 100%|██████████| 6354/6354 [00:00<00:00, 6963.83 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 6354/6354 [00:00<00:00, 27260.76 examples/s]
Dropping fully masked examples from eval dataset: 100%|██████████| 1022/1022 [00:00<00:00, 24395.34 examples/s]


trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


In [8]:
# Référence avant entraînement (adaptateur LoRA initialisé à zéro = modèle de base)
val_before = trainer.evaluate(metric_key_prefix="val_before")
val_before

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


Training Loss,Validation Loss,Step,Before Loss,Entropy,Mean Token Accuracy,Num Tokens
No log,No log,0,0.781484,0.814058,0.771644,0.000000


{'val_before_loss': 0.7814843058586121,
 'eval_entropy': 0.814058095915243,
 'eval_mean_token_accuracy': 0.7716443995013833,
 'eval_num_tokens': 0.0}

In [9]:
# Reprise après interruption : trainer.train(resume_from_checkpoint=True)
train_result = trainer.train()
train_result.metrics

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
100,0.670977,0.654893,0.658708,0.800215,620405.000000
200,0.664538,0.646578,0.641936,0.801951,1239530.000000
300,0.655406,0.643910,0.642732,0.802477,1850397.000000
398,0.673281,0.643092,0.636405,0.802864,2438068.000000


{'train_runtime': 1810.7712,
 'train_samples_per_second': 3.509,
 'train_steps_per_second': 0.22,
 'total_flos': 3.184611716310221e+16,
 'train_loss': 0.667581258706711,
 'epoch': 1.0}

In [10]:
# Sauvegarde de l'adaptateur LoRA (meilleur checkpoint) et de la configuration du run
FINAL_DIR = OUTPUT_DIR / "final"
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

run_config = {
    "model": MODEL_NAME,
    "seed": SEED,
    "small_run": SMALL_RUN,
    "lora": {"r": lora_config.r, "alpha": lora_config.lora_alpha, "dropout": lora_config.lora_dropout},
    "data": {name: len(ds) for name, ds in splits.items()},
    "training_args": training_args.to_dict(),
}
(FINAL_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2, default=str), encoding="utf-8")

4580

## 6. Validation intermédiaire

1. **Loss, perplexité et token accuracy** sur validation (avant / après SFT) et test.
2. **Qualité des réponses générées** sur des exemples de test, modèle de base vs modèle SFT, sur les **lettres** données dans la réponse. Génération par lots pour bien charger le GPU.
   - `accuracy` : exactement les lettres attendues ;
   - `option_accuracy` : exactitude par proposition, part des cinq propositions correctement cochées ou laissées. Utile en `mcqm` ; en `mcqu`, une réponse vide obtient déjà 0,8 : y lire l'accuracy ;
   - `f1` : crédit partiel pour les réponses multiples. Il favorise les réponses qui cochent trop : en `mcqm`, tout cocher donne un F1 d'environ 0,7 ;
   - `extra_letters` : lettres données en trop (positif) ou en moins (négatif) par rapport au corrigé, en moyenne.

   Deux **réponses naïves**, identiques pour toutes les questions, servent de référence : `frequente` (la réponse la plus fréquente du train) et `tout_cocher` (A à E). Un modèle utile doit les battre, sur l'accuracy et l'exactitude par proposition.

La **latence** n'est pas mesurée ici : avec l'adaptateur LoRA non fusionné et Transformers, elle ne serait pas représentative. Elle sera mesurée en phase 3, sur le modèle final fusionné servi par vLLM.

In [11]:
val_after = trainer.evaluate(metric_key_prefix="val_after")
test_metrics = trainer.evaluate(splits["test"], metric_key_prefix="test")

# token_accuracy : part des tokens de la réponse attendue que le modèle prédit correctement
# (TRL la nomme « eval_mean_token_accuracy » quel que soit le préfixe : on la renomme ici)
evaluations = {"val_before": val_before, "val_after": val_after, "test": test_metrics}
summary = pd.DataFrame({
    name: {"loss": m[f"{name}_loss"], "token_accuracy": m["eval_mean_token_accuracy"]}
    for name, m in evaluations.items()
}).T
summary["perplexity"] = summary["loss"].map(math.exp)

# Même mesure sur le test, source par source : chaque format doit être appris
per_source = {}
for source in SFT_DATASETS:
    subset = splits["test"].filter(lambda example: example["source"] == source)
    if len(subset):
        m = trainer.evaluate(subset, metric_key_prefix=f"test_{source}")
        per_source[f"test_{source}"] = {"loss": m[f"test_{source}_loss"], "token_accuracy": m["eval_mean_token_accuracy"]}
per_source_summary = pd.DataFrame(per_source).T
per_source_summary["perplexity"] = per_source_summary["loss"].map(math.exp)
pd.concat([summary, per_source_summary])

Training Loss,Validation Loss,Step,After Loss,Entropy,Mean Token Accuracy,Num Tokens
0.673281,No log,398,0.643092,0.636405,0.802864,2438068.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 2180/2180 [00:00<00:00, 31816.94 examples/s]


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
0.673281,0.588502,398,0.585607,0.816664,2438068.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 749/749 [00:00<00:00, 25165.29 examples/s]


Training Loss,Validation Loss,Step,Sft Mediqal Mcqu Loss,Entropy,Mean Token Accuracy,Num Tokens
0.673281,No log,398,0.113351,0.113788,0.952945,2438068.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 931/931 [00:00<00:00, 36176.89 examples/s]


Training Loss,Validation Loss,Step,Sft Mediqal Mcqm Loss,Entropy,Mean Token Accuracy,Num Tokens
0.673281,No log,398,0.082983,0.077559,0.963018,2438068.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 500/500 [00:00<00:00, 20953.92 examples/s]


Training Loss,Validation Loss,Step,Sft Ultramedical Loss,Entropy,Mean Token Accuracy,Num Tokens
0.673281,No log,398,0.784546,0.785476,0.759954,2438068.000000


,loss,token_accuracy,perplexity
val_before,0.781484,0.771644,2.184713
val_after,0.643092,0.802864,1.902354
test,0.588502,0.816664,1.801287
test_sft_mediqal_mcqu,0.113351,0.952945,1.120025
test_sft_mediqal_mcqm,0.082983,0.963018,1.086523
test_sft_ultramedical,0.784546,0.759954,2.191413


In [4]:
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
STOP_IDS = [IM_END_ID, tokenizer.eos_token_id]
# Réponses mcqu : « C. texte de la proposition ». La lettre vient en premier : une réponse coupée reste évaluable.
# Le modèle de base, qui ne s'arrête pas toujours, est ainsi coupé plus tôt.
MAX_NEW_TOKENS = 64


def render(prompt):
    return tokenizer.apply_chat_template(prompt, add_generation_prompt=True, tokenize=False)


@torch.no_grad()
def generate_batch(prompts, batch_size=16, max_new_tokens=MAX_NEW_TOKENS, desc=None, model=None):
    # Génération par lots (GPU bien chargé) : renvoie les réponses, dans l'ordre des prompts
    model = model or trainer.model  # par défaut, le modèle du SFT
    texts = [render(prompt) for prompt in prompts]
    # Prompts triés par longueur : moins de padding dans chaque lot, les plus longs d'abord
    # (un manque de mémoire GPU apparaît dès le premier lot)
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]), reverse=True)
    answers = [None] * len(texts)
    padding_side, tokenizer.padding_side = tokenizer.padding_side, "left"  # padding à gauche : la génération suit chaque prompt
    try:
        for start in tqdm(range(0, len(order), batch_size), desc=desc, unit="lot"):
            batch = order[start:start + batch_size]
            inputs = tokenizer([texts[i] for i in batch], return_tensors="pt", padding=True, add_special_tokens=False).to(device)
            output = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                eos_token_id=STOP_IDS, pad_token_id=tokenizer.pad_token_id,
            )
            decoded = tokenizer.batch_decode(output[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            for i, answer in zip(batch, decoded):
                answers[i] = answer.strip()
    finally:
        tokenizer.padding_side = padding_side
    return answers


QUALITY_METRICS = ["accuracy", "option_accuracy", "f1", "extra_letters"]
NAIVE_ANSWERS = ["frequente", "tout_cocher"]


def letter_scores(predicted: frozenset, expected: frozenset) -> dict:
    # Mesures sur les lettres d'une réponse (ensembles de lettres A à E)
    common = len(predicted & expected)
    return {
        "accuracy": float(predicted == expected),  # exactement les lettres attendues
        # Exactitude par proposition : part des 5 propositions correctement cochées ou laissées
        "option_accuracy": sum((letter in predicted) == (letter in expected) for letter in LETTERS) / len(LETTERS),
        # F1 : crédit partiel ; favorise les réponses qui cochent trop (tout cocher : ~0,7 en mcqm)
        "f1": 2 * common / (len(predicted) + len(expected)) if common else 0.0,
        "extra_letters": len(predicted) - len(expected),  # > 0 : coche trop ; < 0 : pas assez
    }


def letters_text(letters) -> str:
    # Réponse réduite à ses lettres (« A.\nC. »), lisible par answer_letters
    return "\n".join(f"{letter}." for letter in sorted(letters))


def naive_answers(train_answers: list[str]) -> dict:
    # Réponses naïves, identiques pour toutes les questions : un modèle utile doit les battre
    most_common = Counter(answer_letters(answer) for answer in train_answers).most_common(1)[0][0]
    return {"frequente": letters_text(most_common), "tout_cocher": letters_text(LETTERS)}


def add_scores(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    # Ajoute <mesure>_<colonne> pour chaque colonne de réponses, comparée à df["reference"]
    expected = [answer_letters(reference) for reference in df["reference"]]
    for column in columns:
        scores = pd.DataFrame(
            [letter_scores(answer_letters(answer), e) for answer, e in zip(df[column], expected)], index=df.index
        )
        for metric in QUALITY_METRICS:
            df[f"{metric}_{column}"] = scores[metric]
    return df


def score_table(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    # Moyennes : une ligne par mesure, une colonne par modèle ou réponse naïve
    return pd.DataFrame({column: {metric: df[f"{metric}_{column}"].mean() for metric in QUALITY_METRICS} for column in columns})

In [13]:
N_EVAL = 20 if SMALL_RUN else 200    # exemples de test par source
BATCH_SIZE = 16                      # à baisser en cas de manque de mémoire GPU (OutOfMemoryError)

# Sources évaluées par génération, avec leur nombre maximal de tokens générés.
# Les réponses anglaises (plusieurs centaines de tokens, rédigées) ne se comparent pas mot à mot à une référence :
# elles sont évaluées par la loss et la perplexité (cellule précédente).
GENERATION_SOURCES = {"sft_mediqal_mcqu": MAX_NEW_TOKENS, "sft_mediqal_mcqm": 192}
ANSWER_COLUMNS = ["base", "sft", *NAIVE_ANSWERS]  # modèles, puis réponses naïves de référence

trainer.model.eval()
trainer.model.config.use_cache = True  # accélère la génération

# Génération par lots, modèle de base puis modèle SFT, source par source
source_generations = []
for source, max_new_tokens in GENERATION_SOURCES.items():
    subset = splits["test"].filter(lambda example: example["source"] == source)
    subset = subset.select(range(min(N_EVAL, len(subset))))
    prompts = list(subset["prompt"])
    with trainer.model.disable_adapter():  # modèle de base, sans LoRA
        base_answers = generate_batch(prompts, BATCH_SIZE, max_new_tokens, desc=f"Base {source}")
    sft_answers = generate_batch(prompts, BATCH_SIZE, max_new_tokens, desc=f"SFT {source}")
    train_answers = [c[0]["content"] for c in splits["train"].filter(lambda example: example["source"] == source)["completion"]]
    source_generations.append(pd.DataFrame({
        "source": source,
        "question": [prompt[-1]["content"][-200:] for prompt in prompts],
        "reference": [completion[0]["content"] for completion in subset["completion"]],
        "base": base_answers,
        "sft": sft_answers,
        **naive_answers(train_answers),  # mêmes réponses pour toutes les questions de la source
    }))
generations = add_scores(pd.concat(source_generations, ignore_index=True), ANSWER_COLUMNS)
generations.to_csv(FINAL_DIR / "test_generations.csv", index=False)

# Moyennes sur toutes les sources, puis source par source
comparison = score_table(generations, ANSWER_COLUMNS)
quality_by_source = pd.concat({source: score_table(group, ANSWER_COLUMNS) for source, group in generations.groupby("source")})
display(quality_by_source)
comparison

Filter: 100%|██████████| 6354/6354 [00:00<00:00, 291617.24 examples/s]


base       sft  frequente  tout_cocher
sft_mediqal_mcqm accuracy         0.020000  0.090000   0.040000     0.050000
                 option_accuracy  0.510000  0.604000   0.529000     0.545000
                 f1               0.308095  0.675056   0.576976     0.690774
                 extra_letters   -0.600000  1.010000   0.275000     2.275000
sft_mediqal_mcqu accuracy         0.005000  0.390000   0.255000     0.000000
                 option_accuracy  0.769000  0.756000   0.702000     0.200000
                 f1               0.017500  0.390000   0.255000     0.333333
                 extra_letters   -0.765000  0.000000   0.000000     4.000000

,base,sft,frequente,tout_cocher
accuracy,0.012500,0.240000,0.147500,0.025000
option_accuracy,0.639500,0.680000,0.615500,0.372500
f1,0.162798,0.532528,0.415988,0.512054
extra_letters,-0.682500,0.505000,0.137500,3.137500


In [14]:
# Quelques exemples à relire
pd.set_option("display.max_colwidth", 200)
generations.groupby("source")[["source", "question", "reference", "base", "sft"]].head(3)

,source,question,reference,base,sft
0,sft_mediqal_mcqu,érieur\n\nLe délai habituel de consolidation d'une fracture de la clavicule est :\n\nPropositions :\nA. 1 semaine\nB. 2 semaines\nC. 4 semaines\nD. 8 semaines\nE. 10 semaines\n\nUne seule proposit...,C. 4 semaines,Assistant médical\n\nAssistant médical\n\nAssistant médical\n\nAssistant médical\n\nAssistant médical\n\nAssistant médical\n\nAssistant médical\n\nAssistant médical\n\nAssistant médical\n\nAssista...,C. 4 semaines
1,sft_mediqal_mcqu,"1,40 g/l.\nD. Glycémie A jeun < 0,95 g/l, glycémie post-prandiale A 2 heures < 1,20 g/l.\nE. Glycémie A jeun < 0,90 g/l, glycémie post-prandiale A 2 heures < 1,20 g/l.\n\nUne seule proposition est...","D. Glycémie A jeun < 0,95 g/l, glycémie post-prandiale A 2 heures < 1,20 g/l.","La réponse correcte est : C. Glycémie A jeun < 1,00 g/l, glycémie post-prandiale A 2 heures < 1,40 g/l.\n\nPour un diabète gestationnel, les objectifs glycémiques recommandés sont généra","C. Glycémie A jeun < 1,00 g/l, glycémie post-prandiale A 2 heures < 1,40 g/l."
2,sft_mediqal_mcqu,"ilial\nB. L'IDR à la tuberculine à 4mm\nC. La fièvre et les signes fonctionnels respiratoires\nD. Les propositions A et C exactes\nE. Les propositions A, B, C, D exactes\n\nUne seule proposition e...",D. Les propositions A et C exactes,"La proposition exacte est **D. Les propositions A, B, C, D exactes**.\n\nExplication :\n- **A. La notion de contage familial** : L'enfant a une notion de contage tuberculeux, ce qui est un facteur...",D. Les propositions A et C exactes
200,sft_mediqal_mcqm,toujours une baisse d'acuité visuelle\nD. Impose un traitement d'urgence en cas de forme maligne\nE. Est toujours accompagnée d'un oedème papillaire\n\nPlusieurs propositions sont exactes (au moin...,- B. S'accompagne souvent d'un épaississement des muscles droits inférieurs\n- D. Impose un traitement d'urgence en cas de forme maligne,Tu es un assistant médical expert. Réponds de façon précise et concise..RemoveEmptyEntries\nTu es un assistant médical expert. Réponds de façon précise et concise..RemoveEmptyEntries\nTu es un ass...,- A. Est toujours bilatérale\n- B. S'accompagne souvent d'un épaississement des muscles droits inférieurs\n- D. Impose un traitement d'urgence en cas de forme maligne
201,sft_mediqal_mcqm,en lumière de Wood\nC. Culture sur milieu de Sabouraud\nD. Recherche des anticorps spécifiques\nE. Recherche d'une hypersensibilité cutanée immédiate\n\nPlusieurs propositions sont exactes (au moi...,- A. Prélèvement de squames\n- C. Culture sur milieu de Sabouraud,"Bonjour Monsieur <PERSON>, je suis un assistant médical et je vais vous aider à confirmer le diagnostic de lésion bilatérale prurigineuse inguinocrurale. Voici les propositions exactes pour confir...",- A. Prélèvement de squames\n- B. Examen en lumière de Wood\n- C. Culture sur milieu de Sabouraud
202,sft_mediqal_mcqm,une rétraction lors de la survenue d'une maladie de Basedow\nE. La paralysie de sa commande nerveuse entraîne une ouverture de la fente palpébrale\n\nPlusieurs propositions sont exactes (au moins...,- A. Il s'insère sur le bord du tarse de la paupière supérieure\n- B. Son fonctionnement est conservé en cas de paralysie faciale périphérique\n- D. Il présente une rétraction lors de la survenue ...,Tu es un assistant médical expert. Réponds de façon précise et concise.��态\n��态\nTu es un assistant médical expert. Réponds de façon précise et concise.��态\n��态\nTu es un assistant médical expert....,- A. Il s'insère sur le bord du tarse de la paupière supérieure\n- B. Son fonctionnement est conservé en cas de paralysie faciale périphérique\n- C. Son fonctionnement est altéré en cas de paralys...


In [15]:
# Métriques finales dans MLflow (les loss sont déjà envoyées par le Trainer), puis clôture du run
mlflow.log_metrics({f"{name}_token_accuracy": row["token_accuracy"] for name, row in summary.iterrows()})
mlflow.log_metrics({f"{name}_perplexity": row["perplexity"] for name, row in summary.iterrows()})
mlflow.log_metrics({
    f"test_{metric}_{model}": comparison.loc[metric, model]
    for metric in QUALITY_METRICS
    for model in comparison.columns
})
# Par source : loss, token accuracy et perplexité du test, mesures des réponses générées et des réponses naïves
mlflow.log_metrics({
    f"{name}_{metric}": value for name, row in per_source_summary.iterrows() for metric, value in row.items()
})
mlflow.log_metrics({
    f"test_{source}_{metric}_{column}": value
    for (source, metric), row in quality_by_source.iterrows()
    for column, value in row.items()
})
mlflow.end_run()

🏃 View run sft-lora-r16-v3-seed42 at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/2/runs/dd106c3a525e4d8db729da0c1ab7d65f
🧪 View experiment at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/2


## Lecture des résultats

| Métrique MLflow | Définition | Attendu |
|---|---|---|
| `loss`, `eval_loss` | courbes de loss d'entraînement et de validation | baissent ensemble ; si `eval_loss` remonte : surapprentissage |
| `val_before_loss`, `val_after_loss`, `test_loss` | loss avant / après SFT et sur le test | après < avant ; test proche de val |
| `*_token_accuracy` | part des tokens de la réponse correctement prédits | augmente après SFT |
| `*_perplexity` | exp(loss) | baisse après SFT |
| `test_sft_<source>_loss`, `_token_accuracy`, `_perplexity` | les mêmes mesures sur le test, source par source | chaque source s'améliore, y compris l'anglais |
| `test_accuracy_*` | réponse générée avec exactement les lettres attendues (sources françaises) | sft > base, sft > `frequente` et `tout_cocher` |
| `test_option_accuracy_*` | exactitude par proposition : part des 5 propositions correctement cochées ou laissées | sft > `frequente` et `tout_cocher` en `mcqm` ; en `mcqu`, une réponse vide obtient 0,8 : lire l'accuracy |
| `test_f1_*` | F1 sur les lettres (crédit partiel pour les réponses multiples) | sft > base ; à lire avec prudence, `tout_cocher` obtient ~0,7 en `mcqm` |
| `test_extra_letters_*` | lettres en trop (> 0) ou en moins (< 0) par rapport au corrigé | proche de 0 |
| `test_sft_<source>_<mesure>_*` | les mêmes mesures, source par source | sft bat les réponses naïves pour `mcqu` et `mcqm` |

- La courbe `eval_mean_token_accuracy` mélange toutes les évaluations (TRL ignore le préfixe) : utiliser `*_token_accuracy`.
- Les mesures portent sur les lettres : une réponse sans lettre (format non appris) compte 0.
- `*_frequente`, `*_tout_cocher` : réponses naïves, identiques pour toutes les questions (réponse la plus fréquente du train, A à E). Elles ne dépendent pas du modèle et donnent le niveau à dépasser.
- La latence (TTFT, TPOT, débit) sera mesurée en phase 3, sur le modèle final servi par vLLM.

**Suite** : passer `SMALL_RUN = False` pour l'entraînement complet, puis DPO (section 7).

## 7. Alignement DPO

Le DPO part du **modèle SFT** et lui apprend à préférer la réponse `chosen` à la réponse `rejected` de chaque paire.

- **Modèle de départ** : modèle de base + adaptateur SFT **fusionné** (les poids LoRA sont ajoutés aux poids du modèle). TRL calcule les log-probabilités directement depuis `lm_head`, ce qui impose de fusionner l'adaptateur SFT avant d'en ajouter un nouveau.
- **Nouvel adaptateur LoRA** entraîné par DPO. La **référence** du DPO est le modèle SFT lui-même : TRL l'obtient en désactivant l'adaptateur, sans charger un second modèle en mémoire.
- **Données** : paires françaises (MediQAl `mcqm`, vérifiées par le corrigé) et anglaises (UltraMedical, filtrées sur la qualité), mélangées pour l'entraînement, évaluées séparément par langue.
- **Reprise après redémarrage** : exécuter d'abord la section 1 (configuration), puis cette section. Le tokenizer et l'adaptateur SFT sont relus depuis `SFT_ADAPTER_DIR`. Avant la validation (7.4), exécuter aussi la cellule des fonctions de génération (section 6).
- **Relancer seulement la validation (7.4)**, sans réentraîner : section 1, puis les deux premières cellules de la section 7 (hyperparamètres, données 7.1), la cellule des fonctions de génération (section 6), et enfin la section 7.4. Sa première cellule recharge l'adaptateur DPO sauvegardé.

In [2]:
# Paires DPO de la phase 1, par langue
DPO_DATASETS = {"fr": "dpo_mediqal_mcqm", "en": "dpo_ultramedical"}
DPO_COLUMNS = ["source", "lang", "pair_check", "prompt", "chosen", "rejected"]  # colonnes communes aux deux jeux

# Hyperparamètres DPO
DPO_BETA = 0.1              # écart toléré avec la référence : plus beta est grand, plus le modèle reste proche du SFT
DPO_LEARNING_RATE = 5e-5    # LoRA : ~1 % des poids entraînés ; à 5e-6 (valeur usuelle en DPO complet), les marges restaient quasi nulles
DPO_MAX_LENGTH = 1024       # prompt + réponse, en tokens
DPO_N_EVAL = 20 if SMALL_RUN else 200  # exemples du test français pour la génération (7.4)
DPO_BATCH_SIZE = 16                    # génération par lots : à baisser en cas d'OutOfMemoryError

SFT_ADAPTER_DIR = Path("../outputs") / RUN_NAME / "final"  # adaptateur SFT de la section 5
DPO_RUN_NAME = f"dpo-lora-r16-{DATA_VERSION}-beta{DPO_BETA}-lr{DPO_LEARNING_RATE}-seed{SEED}" + ("-small" if SMALL_RUN else "")
DPO_OUTPUT_DIR = Path("../outputs") / DPO_RUN_NAME  # checkpoints, adaptateur DPO et modèle final fusionné
print(f"SFT : {SFT_ADAPTER_DIR}\nDPO : {DPO_OUTPUT_DIR}")

SFT : ../outputs/sft-lora-r16-v3-seed42/final
DPO : ../outputs/dpo-lora-r16-v3-beta0.1-lr5e-05-seed42


### 7.1 Données DPO

In [3]:
# Tokenizer sauvegardé avec l'adaptateur SFT : même gabarit qu'à l'entraînement SFT
tokenizer = AutoTokenizer.from_pretrained(SFT_ADAPTER_DIR)


def pair_length(example):
    # Longueur en tokens de la plus longue des deux conversations (prompt + chosen, prompt + rejected)
    return max(
        len(tokenizer(tokenizer.apply_chat_template(example["prompt"] + answer, tokenize=False), add_special_tokens=False)["input_ids"])
        for answer in (example["chosen"], example["rejected"])
    )


dpo_by_lang = {}
for lang, stem in DPO_DATASETS.items():
    files = {split: f"../data/anonymized/{stem}_{split}.anonymized.jsonl" for split in ("train", "validation", "test")}
    ds = load_dataset("json", data_files=files)
    # Paires trop longues : TRL couperait la fin de la réponse, on les écarte
    ds = ds.filter(lambda example: pair_length(example) <= DPO_MAX_LENGTH)
    if SMALL_RUN:
        for split, size in {"train": 32, "validation": 8, "test": 8}.items():
            ds[split] = ds[split].shuffle(seed=SEED).select(range(size))
    if lang == "fr":  # type d'erreur de la rejected (omission, ajout, substitution), pour la validation 7.4
        fr_rejected_types = {split: d["rejected_type"] for split, d in ds.items()}
    dpo_by_lang[lang] = ds.select_columns(DPO_COLUMNS)
    print(f"{lang}: " + ", ".join(f"{split} {len(d)}" for split, d in ds.items()))

# Entraînement et validation : les deux langues mélangées
dpo_train = concatenate_datasets([ds["train"] for ds in dpo_by_lang.values()]).shuffle(seed=SEED)
dpo_validation = concatenate_datasets([ds["validation"] for ds in dpo_by_lang.values()]).shuffle(seed=SEED)
print(f"DPO : train {len(dpo_train)}, validation {len(dpo_validation)}")

Filter: 100%|██████████| 481/481 [00:00<00:00, 639.37 examples/s]


fr: train 1357, validation 220, test 481


Filter: 100%|██████████| 756/756 [00:02<00:00, 340.77 examples/s]

en: train 1994, validation 297, test 750
DPO : train 3351, validation 517


In [18]:
# Une paire telle que la verra le modèle
example = dpo_train[0]
print(tokenizer.apply_chat_template(example["prompt"], add_generation_prompt=True, tokenize=False))
print(f"--- CHOSEN :\n{example['chosen'][0]['content']}\n--- REJECTED :\n{example['rejected'][0]['content']}")

<|im_start|>system
You are an expert medical assistant. Answer accurately and clearly.<|im_end|>
<|im_start|>user
A person developed anterior wall MI and died within 2 hours of onset of the symptoms. The pathological autopsy of the hea and observation of the involved myocardium under light microscopy, is underway. What will the finding be, if any?

A. Beginning of coagulation necrosis
B. Neutrophilic infiltration
C. Waviness of myosite fibres at the border
D. Well developed phagocytosis of dead cells<|im_end|>
<|im_start|>assistant

--- CHOSEN :
Let's think step by step.

The patient died within 2 hours of onset of symptoms, which means the infarction is very acute. In the first 2 hours, there will be no visible changes on light microscopy. The earliest changes of myocardial infarction, such as coagulation necrosis, will not be apparent until around 4-6 hours after the onset of ischemia.

However, there will be some subtle changes that can be observed, particularly at the border of the

### 7.2 Modèle de départ : modèle SFT fusionné

In [19]:
# Libère la mémoire GPU du SFT si la section 5 vient de tourner
if "trainer" in globals():
    del trainer, model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

# Modèle de base + adaptateur SFT fusionné : point de départ et référence du DPO
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype)
sft_model = PeftModel.from_pretrained(base_model, str(SFT_ADAPTER_DIR)).merge_and_unload().to(device)
print(f"{sft_model.num_parameters() / 1e9:.2f} B paramètres")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1712.33it/s]


1.72 B paramètres


### 7.3 Entraînement DPO

- **LoRA** : même configuration que pour le SFT, sur un nouvel adaptateur.
- **Une époque** : le DPO sur-apprend vite. Au-delà, le modèle mémorise les paires sans mieux généraliser.
- **Batch effectif de 16** (2 × 8) : chaque exemple contient deux réponses, donc deux fois plus de tokens qu'en SFT.
- **Learning rate de 5e-5**, avec montée progressive sur 10 % des étapes. Plus élevé qu'en DPO complet (5e-6 à 5e-7) : avec LoRA, seule une petite partie des poids est entraînée. Au premier run, 5e-6 laissait la marge des récompenses quasi nulle sur le français et la courbe n'avait pas atteint son plateau. Si `logps/chosen` chute fortement, revenir à 2e-5.

In [20]:
dpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

# Suivi MLflow : même serveur que le SFT (connexion faite en section 5), expérience dédiée au DPO
if MLFLOW_BACKEND == "dagshub":
    import dagshub

    dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
else:
    mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("qwen3-1.7b-dpo-lora")
mlflow.start_run(run_name=DPO_RUN_NAME)
mlflow.log_params({
    "model": MODEL_NAME, "data_version": DATA_VERSION, "sft_adapter": str(SFT_ADAPTER_DIR), "small_run": SMALL_RUN, "device": device,
    "beta": DPO_BETA, "lora_r": dpo_lora_config.r, "lora_alpha": dpo_lora_config.lora_alpha,
    **{f"pairs_{lang}_{split}": len(d) for lang, ds in dpo_by_lang.items() for split, d in ds.items()},
})

dpo_args = DPOConfig(
    output_dir=str(DPO_OUTPUT_DIR),
    run_name=DPO_RUN_NAME,
    seed=SEED,
    # DPO
    beta=DPO_BETA,
    loss_type=["sigmoid"],  # perte DPO standard
    max_length=DPO_MAX_LENGTH,
    # Optimisation
    num_train_epochs=1,
    max_steps=10 if SMALL_RUN else -1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,  # batch effectif = 16 paires
    learning_rate=DPO_LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,  # 10 % des étapes en montée progressive du learning rate
    gradient_checkpointing=True,
    bf16=bf16,
    fp16=dtype == torch.float16,
    # Évaluation et checkpoints
    eval_strategy="steps",
    eval_steps=5 if SMALL_RUN else 25,
    save_steps=5 if SMALL_RUN else 25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # Logs
    logging_steps=1 if SMALL_RUN else 5,
    report_to="mlflow",
)

dpo_trainer = DPOTrainer(
    model=sft_model,
    args=dpo_args,
    train_dataset=dpo_train,
    eval_dataset=dpo_validation,
    processing_class=tokenizer,
    peft_config=dpo_lora_config,
)
dpo_trainer.model.print_trainable_parameters()

Initialized MLflow to track repo "obiyohan/my-ml-mlflow"

Repository obiyohan/my-ml-mlflow initialized!

Tokenizing train dataset: 100%|██████████| 3351/3351 [00:11<00:00, 298.04 examples/s]
Dropping fully truncated examples from train dataset: 100%|██████████| 3351/3351 [00:00<00:00, 34252.86 examples/s]
Tokenizing eval dataset: 100%|██████████| 517/517 [00:01<00:00, 274.72 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 517/517 [00:00<00:00, 26216.94 examples/s]

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


In [21]:
# Reprise après interruption : dpo_trainer.train(resume_from_checkpoint=True)
dpo_result = dpo_trainer.train()
dpo_result.metrics

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
25,0.616375,0.619459,0.666541,372295.000000,4.416251,4.488374,0.797733,-0.083980,-0.276579,0.698077,0.192599,-161.739724,-150.832134
50,0.578591,0.519843,0.764840,743789.000000,4.478565,4.634679,0.788708,-1.123615,-2.327046,0.742308,1.203431,-172.136071,-171.336803
75,0.506266,0.488287,0.705190,1111986.000000,3.817027,3.986955,0.790175,-1.110753,-2.463487,0.734615,1.352735,-172.007450,-172.701218
100,0.413564,0.468990,0.747260,1470614.000000,3.953877,4.132342,0.787431,-1.358744,-2.840320,0.755769,1.481576,-174.487360,-176.469541
125,0.389012,0.458953,0.759877,1842249.000000,4.103660,4.286322,0.785735,-1.556700,-3.093257,0.765385,1.536557,-176.466923,-178.998920
150,0.446491,0.446540,0.754626,2214597.000000,4.016981,4.205334,0.785542,-1.601529,-3.267941,0.778846,1.666412,-176.915213,-180.745757
175,0.381662,0.447762,0.759384,2587479.000000,4.010529,4.201380,0.784649,-1.734074,-3.467631,0.784615,1.733557,-178.240661,-182.742658
200,0.420656,0.449608,0.760555,2970182.000000,4.016045,4.206422,0.784576,-1.731510,-3.453272,0.790385,1.721762,-178.215028,-182.599069
210,0.508106,0.446014,0.760572,3112177.000000,4.017403,4.206612,0.784716,-1.725106,-3.452857,0.788462,1.727751,-178.150984,-182.594911


{'train_runtime': 3158.358,
 'train_samples_per_second': 1.061,
 'train_steps_per_second': 0.066,
 'total_flos': 3.405964661415936e+16,
 'train_loss': 0.4705430059205918,
 'epoch': 1.0}

In [22]:
# Sauvegarde de l'adaptateur DPO (meilleur checkpoint) et de la configuration du run
DPO_FINAL_DIR = DPO_OUTPUT_DIR / "final"
dpo_trainer.save_model(str(DPO_FINAL_DIR))
tokenizer.save_pretrained(str(DPO_FINAL_DIR))

dpo_run_config = {
    "model": MODEL_NAME,
    "sft_adapter": str(SFT_ADAPTER_DIR),  # l'adaptateur DPO s'applique au modèle SFT fusionné
    "seed": SEED,
    "small_run": SMALL_RUN,
    "beta": DPO_BETA,
    "lora": {"r": dpo_lora_config.r, "alpha": dpo_lora_config.lora_alpha, "dropout": dpo_lora_config.lora_dropout},
    "data": {f"{lang}_{split}": len(d) for lang, ds in dpo_by_lang.items() for split, d in ds.items()},
    "training_args": dpo_args.to_dict(),
}
(DPO_FINAL_DIR / "run_config.json").write_text(json.dumps(dpo_run_config, indent=2, default=str), encoding="utf-8")

4897

### 7.4 Validation DPO

1. **Préférences** sur la validation et le test de chaque langue, et sur deux découpages du test :
   - **anglais, selon la longueur** (`chosen` plus longue ou plus courte que la `rejected`) : une précision très différente entre les deux révèle une préférence apprise pour la longueur plutôt que pour le fond ;
   - **français, selon le type d'erreur de la `rejected`** : *omission* (une bonne proposition retirée), *ajout* (une mauvaise ajoutée), *substitution* (l'une remplacée par l'autre). Un modèle qui a appris « cocher moins » plutôt que « cocher juste » réussit les ajouts et rate les omissions.

   La *récompense implicite* d'une réponse mesure combien le modèle DPO la juge plus probable que le modèle SFT. On en tire : 
   - `reward_accuracy` : part des paires où la récompense de la `chosen` dépasse celle de la `rejected` (0,5 = hasard) ;
   - `reward_margin` : écart moyen entre les deux récompenses ;
   - `logps_chosen` / `logps_rejected` : log-probabilités moyennes des deux réponses sous le modèle DPO.
2. **Réponses générées** sur le test français : modèle SFT vs modèle DPO, avec les mesures et les réponses naïves de la section 6, le décompte des questions où un seul des deux modèles a la bonne réponse, et les résultats selon le nombre de bonnes réponses attendues (2 à 5). Le test `mcqm` a un corrigé : on compare chaque réponse à la liste des bonnes réponses.

In [5]:
# Modèle DPO pour la validation : celui de 7.3 s'il est encore en mémoire, sinon l'adaptateur sauvegardé.
# Après 7.5, l'adaptateur DPO est fusionné dans les poids : désactiver l'adaptateur ne donnerait plus le modèle SFT
# (référence du DPO) et toutes les récompenses seraient nulles. Il faut alors recharger.
from peft import set_peft_model_state_dict
from safetensors.torch import load_file

DPO_FINAL_DIR = DPO_OUTPUT_DIR / "final"
dpo_in_memory = "dpo_trainer" in globals() and any("lora_" in name for name, _ in dpo_trainer.model.named_parameters())
if not dpo_in_memory:
    for name in ("dpo_trainer", "dpo_model", "merged_model", "sft_model", "base_model", "trainer", "model"):
        globals().pop(name, None)
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    # Comme en 7.2 et 7.3 : modèle SFT fusionné + nouvel adaptateur ; la référence reste le modèle SFT (adaptateur désactivé).
    # Ne pas passer un PeftModel déjà chargé : TRL prendrait une copie de l'adaptateur DPO comme référence.
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype)
    sft_model = PeftModel.from_pretrained(base_model, str(SFT_ADAPTER_DIR)).merge_and_unload().to(device)
    eval_args = DPOConfig(
        output_dir=str(DPO_OUTPUT_DIR / "eval"), seed=SEED, beta=DPO_BETA, loss_type=["sigmoid"], max_length=DPO_MAX_LENGTH,
        per_device_eval_batch_size=4, bf16=bf16, fp16=dtype == torch.float16, report_to="none",
    )
    dpo_trainer = DPOTrainer(
        model=sft_model, args=eval_args, train_dataset=dpo_validation.select(range(1)),  # train_dataset exigé par TRL, inutilisé
        processing_class=tokenizer, peft_config=LoraConfig.from_pretrained(str(DPO_FINAL_DIR)),
    )
    # Poids de l'adaptateur DPO sauvegardé en 7.3, dans le nouvel adaptateur
    result = set_peft_model_state_dict(dpo_trainer.model, load_file(str(DPO_FINAL_DIR / "adapter_model.safetensors")))
    assert not result.unexpected_keys, result.unexpected_keys[:5]
    assert not [key for key in result.missing_keys if "lora_" in key], "poids LoRA manquants"
print(f"Modèle DPO : {'en mémoire (7.3)' if dpo_in_memory else f'rechargé depuis {DPO_FINAL_DIR}'}")

Tokenizing train dataset: 100%|██████████| 1/1 [00:00<00:00, 59.05 examples/s]
Dropping fully truncated examples from train dataset: 100%|██████████| 1/1 [00:00<00:00, 165.75 examples/s]


Modèle DPO : rechargé depuis ../outputs/dpo-lora-r16-v3-beta0.1-lr5e-05-seed42/final


In [6]:
# 1. Préférences, par split et par langue
# (TRL nomme ses métriques « eval_rewards/... » quel que soit le préfixe : on les renomme ici)
eval_sets = {f"{split}_{lang}": ds[split] for lang, ds in dpo_by_lang.items() for split in ("validation", "test")}

# Test anglais selon la longueur : la chosen est-elle plus longue ou plus courte que la rejected ?
test_en = dpo_by_lang["en"]["test"]
chosen_longer = [len(c[0]["content"]) > len(r[0]["content"]) for c, r in zip(test_en["chosen"], test_en["rejected"])]
eval_sets["test_en_chosen_longer"] = test_en.select([i for i, longer in enumerate(chosen_longer) if longer])
eval_sets["test_en_chosen_shorter"] = test_en.select([i for i, longer in enumerate(chosen_longer) if not longer])

# Test français selon le type d'erreur de la rejected
for rejected_type in ("omission", "ajout", "substitution"):
    indices = [i for i, t in enumerate(fr_rejected_types["test"]) if t == rejected_type]
    eval_sets[f"test_fr_{rejected_type}"] = dpo_by_lang["fr"]["test"].select(indices)

dpo_rows = {}
for name, dataset in eval_sets.items():
    if len(dataset) == 0:  # petit run : un sous-ensemble peut être vide
        continue
    m = dpo_trainer.evaluate(dataset, metric_key_prefix=name)
    dpo_rows[name] = {
        "pairs": len(dataset),
        "loss": m[f"{name}_loss"],
        "reward_accuracy": m["eval_rewards/accuracies"],
        "reward_margin": m["eval_rewards/margins"],
        "logps_chosen": m["eval_logps/chosen"],
        "logps_rejected": m["eval_logps/rejected"],
    }
dpo_summary = pd.DataFrame(dpo_rows).T
dpo_summary

Tokenizing eval dataset: 100%|██████████| 220/220 [00:00<00:00, 371.14 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 220/220 [00:00<00:00, 29340.12 examples/s]
/home/johan/projects/P14/.venv/lib/python3.14/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


Training Loss,Validation Loss,Step,Fr Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
No log,No log,0,0.548569,0.024903,0.000000,-0.772889,-0.632690,0.957736,-0.705146,-1.511245,0.718182,0.806099,-9.968340,-19.268283


Tokenizing eval dataset: 100%|██████████| 481/481 [00:01<00:00, 432.12 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 481/481 [00:00<00:00, 38541.60 examples/s]


Training Loss,Validation Loss,Step,Fr Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
No log,No log,0,0.524519,0.028897,0.000000,-0.659342,-0.543669,0.953858,-0.674425,-1.584944,0.704545,0.910519,-9.750105,-20.214014


Tokenizing eval dataset: 100%|██████████| 297/297 [00:01<00:00, 257.63 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 297/297 [00:00<00:00, 20109.58 examples/s]


Training Loss,Validation Loss,Step,En Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
No log,No log,0,0.369423,0.870440,0.000000,4.737523,4.893621,0.757290,-2.505960,-4.935646,0.840000,2.429686,-303.398659,-304.235876


... (9 outputs left out)

Training Loss,Validation Loss,Step,Fr Ajout Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
No log,No log,0,0.417105,0.029333,0.000000,-0.555405,-0.330990,0.951867,-0.638699,-1.475353,0.855263,0.836654,-9.385205,-18.697101


Tokenizing eval dataset: 100%|██████████| 155/155 [00:00<00:00, 431.91 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 155/155 [00:00<00:00, 27480.96 examples/s]


Training Loss,Validation Loss,Step,Fr Substitution Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
No log,No log,0,0.641159,0.028852,0.000000,-0.804352,-0.723811,0.953781,-0.601021,-0.849299,0.608974,0.248278,-8.989644,-11.955515


,pairs,loss,reward_accuracy,reward_margin,logps_chosen,logps_rejected
validation_fr,220.0,0.548569,0.718182,0.806099,-9.968340,-19.268283
test_fr,481.0,0.524519,0.704545,0.910519,-9.750105,-20.214014
validation_en,297.0,0.369423,0.840000,2.429686,-303.398659,-304.235876
test_en,750.0,0.707980,0.656915,0.860384,-366.811600,-277.644357
test_en_chosen_longer,524.0,0.773092,0.622137,0.485309,-397.978207,-250.258987
test_en_chosen_shorter,226.0,0.547030,0.767544,1.774362,-295.597113,-342.308376
test_fr_omission,176.0,0.512559,0.636364,1.576149,-10.765680,-28.995905
test_fr_ajout,150.0,0.417105,0.855263,0.836654,-9.385205,-18.697101
test_fr_substitution,155.0,0.641159,0.608974,0.248278,-8.989644,-11.955515


In [7]:
# 2. Réponses générées sur le test français : modèle SFT (adaptateur DPO désactivé) vs modèle DPO
assert "generate_batch" in globals(), "Exécuter d'abord la cellule des fonctions de génération (section 6)"
DPO_MAX_NEW_TOKENS = 192  # réponses mcqm : listes de 2 à 5 options
DPO_ANSWER_COLUMNS = ["sft", "dpo", *NAIVE_ANSWERS]

dpo_model = dpo_trainer.model.eval()
dpo_model.config.use_cache = True  # accélère la génération

test_fr = dpo_by_lang["fr"]["test"].select(range(min(DPO_N_EVAL, len(dpo_by_lang["fr"]["test"]))))
fr_prompts = list(test_fr["prompt"])
fr_references = [chosen[0]["content"] for chosen in test_fr["chosen"]]

with dpo_model.disable_adapter():
    fr_sft_answers = generate_batch(fr_prompts, DPO_BATCH_SIZE, DPO_MAX_NEW_TOKENS, desc="Génération SFT", model=dpo_model)
fr_dpo_answers = generate_batch(fr_prompts, DPO_BATCH_SIZE, DPO_MAX_NEW_TOKENS, desc="Génération DPO", model=dpo_model)

dpo_generations = add_scores(pd.DataFrame({
    "question": [prompt[-1]["content"][-200:] for prompt in fr_prompts],
    "reference": fr_references,
    "sft": fr_sft_answers,
    "dpo": fr_dpo_answers,
    **naive_answers([chosen[0]["content"] for chosen in dpo_by_lang["fr"]["train"]["chosen"]]),
}), DPO_ANSWER_COLUMNS)
dpo_generations["expected_letters"] = [len(answer_letters(reference)) for reference in fr_references]
dpo_generations.to_csv(DPO_FINAL_DIR / "test_fr_generations.csv", index=False)

# Selon le nombre de bonnes réponses attendues : un modèle qui répond toujours 2 lettres réussit seulement la 1re ligne
by_expected = dpo_generations.groupby("expected_letters")[
    [f"{metric}_{model}" for metric in ("accuracy", "extra_letters") for model in ("sft", "dpo")]
].mean()
by_expected.insert(0, "questions", dpo_generations.groupby("expected_letters").size())
display(by_expected)

# Questions où un seul des deux modèles a exactement la bonne réponse (test de McNemar : ce sont elles qui comptent)
only_dpo = int(((dpo_generations["accuracy_dpo"] == 1) & (dpo_generations["accuracy_sft"] == 0)).sum())
only_sft = int(((dpo_generations["accuracy_sft"] == 1) & (dpo_generations["accuracy_dpo"] == 0)).sum())
print(f"Bonne réponse du DPO seul : {only_dpo} questions ; du SFT seul : {only_sft} questions")

dpo_comparison = score_table(dpo_generations, DPO_ANSWER_COLUMNS)
dpo_comparison

Génération DPO: 100%|██████████| 13/13 [02:06<00:00,  9.72s/lot]


,questions,accuracy_sft,accuracy_dpo,extra_letters_sft,extra_letters_dpo
expected_letters,,,,,
2,87,0.126437,0.298851,1.333333,0.160920
3,76,0.118421,0.092105,0.789474,-0.684211
4,30,0.133333,0.066667,-0.300000,-1.700000
5,7,0.428571,0.000000,-0.714286,-2.857143


Bonne réponse du DPO seul : 22 questions ; du SFT seul : 14 questions


,sft,dpo,frequente,tout_cocher
accuracy,0.135000,0.17500,0.040000,0.035000
option_accuracy,0.642000,0.63500,0.487000,0.557000
f1,0.703187,0.63175,0.452833,0.701905
extra_letters,0.810000,-0.54500,-0.785000,2.215000


In [8]:
# Quelques exemples à relire
dpo_generations[["question", "reference", "sft", "dpo"]].head(5)

,question,reference,sft,dpo
0,e pas augmenter les apports hydriques quand on...,- D. Ne pas augmenter les apports hydriques qu...,- A. Commencer par de faibles doses de diuréti...,- A. Commencer par de faibles doses de diuréti...
1,Crises exclusivement spontanées\nB. Crises no...,- A. Crises exclusivement spontanées\n- B. Cri...,- A. Crises exclusivement spontanées\n- B. Cri...,- B. Crises nocturnes à horaire fixe\n- D. Evo...
2,e(s) suivante(s) :\n\nPropositions :\nA. Crise...,- B. Troubles de la conscience\n- C. Douleurs ...,- A. Crises convulsives\n- B. Troubles de la c...,- A. Crises convulsives\n- B. Troubles de la c...
3,nterne\nD. Deux microgéodes sous-chondrales au...,"- A. Pincement d'environ 50 %, localisé a l'in...","- A. Pincement d'environ 50 %, localisé a l'in...","- A. Pincement d'environ 50 %, localisé a l'in..."
4,héliale\nB. Présence de ponts d'union\nC. Prés...,- A. Prolifération épithéliale\n- D. Envahisse...,- A. Prolifération épithéliale\n- B. Présence ...,- B. Présence de ponts d'union\n- C. Présence ...


In [9]:
# Métriques finales dans MLflow, puis clôture du run
# Validation relancée seule : les métriques rejoignent le run DPO existant (même nom), au lieu d'en créer un nouveau
if mlflow.active_run() is None:
    if MLFLOW_BACKEND == "dagshub":
        import dagshub

        dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
    else:
        mlflow.set_tracking_uri("sqlite:///../mlflow.db")
    previous = mlflow.search_runs(
        experiment_names=["qwen3-1.7b-dpo-lora"], filter_string=f"tags.mlflow.runName = '{DPO_RUN_NAME}'",
        order_by=["start_time DESC"], max_results=1,
    )
    assert len(previous), f"run {DPO_RUN_NAME} introuvable dans MLflow"
    mlflow.start_run(run_id=previous["run_id"].iloc[0])
mlflow.log_metrics({
    f"{name}_{metric}": value
    for name, row in dpo_summary.iterrows()
    for metric, value in row.items()
})
mlflow.log_metrics({
    f"test_fr_{metric}_{model}": dpo_comparison.loc[metric, model]
    for metric in QUALITY_METRICS
    for model in dpo_comparison.columns
})
mlflow.log_metrics({"test_fr_only_dpo_correct": only_dpo, "test_fr_only_sft_correct": only_sft})
mlflow.end_run()

Accessing as ObiYohan

Initialized MLflow to track repo "obiyohan/my-ml-mlflow"

Repository obiyohan/my-ml-mlflow initialized!

🏃 View run dpo-lora-r16-v3-beta0.1-lr5e-05-seed42 at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/3/runs/ec6f47a11dc94763a07fd09033dbfdcf
🧪 View experiment at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/3


### 7.5 Modèle final fusionné

Modèle de base + SFT + DPO en un seul jeu de poids complets (environ 3,4 Go en bf16), avec le tokenizer et son gabarit : c'est le modèle à servir avec vLLM en phase 3.

In [27]:
MERGED_DIR = DPO_OUTPUT_DIR / "merged"
merged_model = dpo_trainer.model.merge_and_unload()
merged_model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))
print(f"Modèle final : {MERGED_DIR}")

Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.26s/it]

Modèle final : ../outputs/dpo-lora-r16-v3-beta0.1-lr5e-05-seed42/merged


## Lecture des résultats DPO

| Métrique MLflow | Définition | Attendu |
|---|---|---|
| `loss`, `eval_loss` | perte DPO d'entraînement et de validation (0,69 = aucune préférence apprise) | baissent sous 0,69 ; si `eval_loss` remonte : surapprentissage |
| `rewards/accuracies` | part des paires où la `chosen` l'emporte sur la `rejected` | monte nettement au-dessus de 0,5 |
| `rewards/margins` | écart moyen de récompense entre `chosen` et `rejected` | augmente, sans exploser |
| `logps/chosen`, `logps/rejected` | log-probabilités moyennes des deux réponses | `rejected` baisse plus que `chosen` ; si `chosen` chute fortement, baisser le learning rate ou augmenter beta |
| `{validation,test}_{fr,en}_*` | les mêmes métriques, par split et par langue | proches entre validation et test, et entre langues |
| `test_en_chosen_{longer,shorter}_*` | test anglais, selon que la `chosen` est la plus longue ou la plus courte | précisions proches ; un grand écart = préférence pour la longueur |
| `test_fr_{omission,ajout,substitution}_*` | test français, selon le type d'erreur de la `rejected` | précisions proches ; ajout ≫ omission = le modèle a appris à cocher moins |
| `test_fr_{accuracy,option_accuracy,f1,extra_letters}_*` | lettres des réponses générées sur le test `mcqm`, comparées au corrigé (mêmes mesures et réponses naïves qu'en section 6) | dpo ≥ sft, et au-dessus de `frequente` et `tout_cocher` |
| `test_fr_only_{dpo,sft}_correct` | questions où un seul des deux modèles a exactement la bonne réponse | only_dpo > only_sft |

- Les courbes `eval_rewards/*` mélangent toutes les évaluations (TRL ignore le préfixe) : pour la comparaison finale, utiliser `{validation,test}_{fr,en}_*`.
- **Itérer** : un run par jeu d'hyperparamètres (`DPO_BETA`, `DPO_LEARNING_RATE`), avec le même `SEED`. Le nom du run les contient.

**Suite** : contrôles de sécurité (hallucinations, recommandations dangereuses) sur le modèle final, puis déploiement avec vLLM (phase 3).